In [1]:
import pandas as pd

# Load combined dataset
df = pd.read_csv("../data/combined.csv")
print(df.shape)
print(df.head())
print(df['label'].value_counts())


(183622, 2)
                                                text  label
0  new book language planning michael clyne edito...      0
1  lucinda burkett lucindaiesafr pamela anderson ...      1
2  sav e hundreds month preferred mor tgage ge mo...      1
3  eol data july 13 2001 eol deals 7 1 2001 7 13 ...      0
4  Check Your Account information. _ PayPalSecure...      1
label
1    93110
0    90512
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")


Train: 132207 | Val: 14690 | Test: 36725


In [3]:
from transformers import BertTokenizer

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_texts = list(map(str, train_texts))
val_texts   = list(map(str, val_texts))
test_texts  = list(map(str, test_texts))

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings   = tokenizer(val_texts, truncation=True, padding=True, max_length=256)
test_encodings  = tokenizer(test_texts, truncation=True, padding=True, max_length=256)



In [4]:
import torch

class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels.iloc[idx]))
        return item

train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)
test_dataset = EmailDataset(test_encodings, test_labels)


In [5]:
from transformers import BertForSequenceClassification

# Load BERT model for binary classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)   # predicted class
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [7]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,        
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipykernel_1334/3831661282.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [8]:
trainer.train()

model.save_pretrained("../models/saved_model")
tokenizer.save_pretrained("../models/saved_model")


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.035000,0.015975,0.995303,0.994638,0.996107,0.995372
2,0.009400,0.028418,0.994418,0.989892,0.999195,0.994522
3,0.000700,0.026632,0.995711,0.992138,0.999463,0.995787
4,0.000600,0.016338,0.997209,0.996249,0.998255,0.997251
5,0.005000,0.016844,0.997209,0.995983,0.998523,0.997251


('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/vocab.txt',
 './saved_model/added_tokens.json')

In [9]:
results = trainer.evaluate(test_dataset)
print("Test Results:", results)


Test Results: {'eval_loss': 0.015732118859887123, 'eval_accuracy': 0.9974404356705242, 'eval_precision': 0.9964630225080385, 'eval_recall': 0.9984964021050371, 'eval_f1': 0.9974786760366933, 'eval_runtime': 197.1219, 'eval_samples_per_second': 186.306, 'eval_steps_per_second': 11.648, 'epoch': 5.0}
